# Linear regression: from chemistry to a tested estimator

**Question.** Can the other recorded chemistry measurements estimate alcohol concentration for a held-out wine sample?

This is an instructional prediction question, not a claim that the features cause alcohol concentration. We will explore the target, define least squares, run a student-owned gradient-descent implementation through the package evaluation boundary, compare it with a trusted solver, and diagnose residual error.

**What this example models.** A real prediction question motivates the least-squares objective; mathematics motivates tested package code; held-out error and residual structure determine what can be claimed.

**Your project version.** Ask a meaningful quantitative question on your data, implement and test the estimator in the package, choose a leakage-resistant split and baseline, compare with a trusted solver, diagnose failures, and decide whether linear structure is adequate. The chemistry conclusions and plots are not reusable answers.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

from dsm_project.algorithms import create_student_algorithm
from dsm_project.config import load_config
from dsm_project.data import load_project_dataset
from dsm_project.experiments import (
    evaluate_supervised_estimator,
    fit_supervised_candidate,
)

config = load_config("configs/example.toml")
dataset = load_project_dataset(config)
task = dataset.tasks["regression"]
frame = dataset.observations
target = task.target_column

print(task)
display(frame[[*task.feature_columns[:4], target]].head())

## Explore the signal before fitting

The target distribution determines what a constant baseline would do and whether a few extreme observations dominate squared error. Pairwise correlation is only a linear, marginal summary; it can suggest useful plots but does not settle feature value, causation, or multivariable performance.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
frame[target].plot.hist(bins=18, ax=axes[0], color="#1b9e77", edgecolor="white")
axes[0].set(title="Alcohol concentration in the sample", xlabel="alcohol", ylabel="sample count")

correlations = (
    frame[[*task.feature_columns, target]]
    .corr(numeric_only=True)[target]
    .drop(target)
    .sort_values()
)
correlations.plot.barh(ax=axes[1], color="#7570b3")
axes[1].set(title="Marginal correlation with alcohol", xlabel="Pearson correlation")
plt.tight_layout();

## Mathematical model

For sample $i$, features $x_i\in\mathbb{R}^p$, weights $w\in\mathbb{R}^p$, and intercept $b\in\mathbb{R}$, the affine prediction is

$$
\widehat y_i = w^\top x_i+b.
$$

The transparent estimator minimizes half mean squared error,

$$
J(w,b)=\frac{1}{2n}\sum_{i=1}^{n}(w^\top x_i+b-y_i)^2.
$$

Its batch gradients are

$$
\nabla_w J=\frac{1}{n}X^\top(Xw+b\mathbf 1-y),
\qquad
\frac{\partial J}{\partial b}=\frac{1}{n}\mathbf 1^\top(Xw+b\mathbf 1-y).
$$

Gradient descent subtracts a positive step size times these first derivatives. Scaling is fitted on training data inside a pipeline because differently sized coordinates otherwise distort the optimization geometry. Least squares estimates conditional means under the specified features; it does not make residuals disappear or prove a causal law.

In [ ]:
student_pipeline = make_pipeline(
    StandardScaler(),
    create_student_algorithm(
        "batch_linear_regression",
        learning_rate=0.05,
        max_iterations=8_000,
    ),
)
student = evaluate_supervised_estimator(
    dataset,
    config,
    task_name="regression",
    estimator_name="student:batch_linear_regression",
    estimator=student_pipeline,
)
reference = fit_supervised_candidate(
    dataset,
    config,
    task_name="regression",
    model_name="linear_regression",
)

evidence = pd.DataFrame(
    [
        {"implementation": student.model_name, student.metric_name: student.metric_value},
        {"implementation": reference.model_name, reference.metric_name: reference.metric_value},
    ]
)
display(evidence)
print("Train/holdout rows:", student.train_rows, student.test_rows)

## Why compare student code with a trusted solver?

Agreement is a differential test of the objective, scaling convention, intercept, split, and prediction rule. It does not prove either implementation is correct for every input. The package sends both estimators through the same split and metric boundary so an accidental difference in evaluation cannot masquerade as an algorithm difference.

Root mean squared error has the target's units and penalizes large residuals strongly. It needs context from a mean baseline, target range, and residual diagnostics.

In [ ]:
observed = student.observed.astype(float)
predicted = student.predictions.astype(float)
residual = observed - predicted

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
axes[0].scatter(observed, predicted, alpha=0.75, color="#1f77b4")
limits = [min(observed.min(), predicted.min()), max(observed.max(), predicted.max())]
axes[0].plot(limits, limits, "--", color="black")
axes[0].set(title="Held-out predictions", xlabel="observed alcohol", ylabel="predicted alcohol")

axes[1].scatter(predicted, residual, alpha=0.75, color="#d95f02")
axes[1].axhline(0, linestyle="--", color="black")
axes[1].set(title="Residual structure", xlabel="predicted alcohol", ylabel="observed - predicted")

optimizer = student.estimator.named_steps["batchlinearregressor"]
axes[2].plot(optimizer.loss_history_, color="#1b9e77")
axes[2].set(title="Training objective", xlabel="batch update", ylabel="half mean squared error")
plt.tight_layout();

print("Mean residual:", residual.mean())
print("Largest absolute residual:", np.abs(residual).max())

## Conclusion, limits, and project work

The student and trusted implementations should produce comparable holdout error under the same contract. A decreasing training loss shows that the optimizer moved downhill; only held-out evidence speaks to generalization. Residual patterns can reveal nonlinear structure, changing variance, influential observations, or missing variables.

For your project, replace the example target and adapter, document target units and split dependence, implement mathematical and failure-mode tests, add a mean baseline, and report uncertainty across appropriate resamples. Keep the reusable estimator in `src/dsm_project/algorithms/student/`; the notebook should explain and diagnose it rather than define it.